# Hybrid mechanistic and machine learning ensemble

This notebook runs the bioreactor project and creates the submission CSV files.

Use Python 3.11 or newer. Python 3.12 is a good choice for this package.

## Step 1. Find the project and choose the mode

Leave `MODE = "regenerate"` for the quick run. This uses the saved predictions for the supplied challenge data and creates the CSV files again.

Change the same setting to `MODE = "retrain"` only when you want to fit the models again. Training can take a long time. It uses the original modeling and validation recipe, including the validation limitations discussed in the guide. Its recorded error is not a confirmed leaderboard result.

For local Jupyter, extract the ZIP and open this notebook inside the `bioreactor_ready_to_run` folder. In Google Colab, open this notebook and upload `bioreactor_ready_to_run.zip` when the next cell asks for it.

Run the cells in order. The notebook generates files without submitting them to the leaderboard.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import io
import json
import subprocess
import sys

MODE = "regenerate"  # Choose "regenerate" or "retrain" here.

if MODE not in {"regenerate", "retrain"}:
    raise ValueError('MODE must be "regenerate" or "retrain".')
if sys.version_info < (3, 11):
    raise RuntimeError("Please use Python 3.11 or newer for this package.")

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

def find_project():
    for candidate in (Path.cwd(), Path.cwd() / "bioreactor_ready_to_run"):
        required = ("run.py", "requirements.txt", "lab/common.py")
        if all((candidate / name).is_file() for name in required):
            return candidate.resolve()
    return None

ROOT = find_project()
if ROOT is None and colab_files is not None:
    print("Upload bioreactor_ready_to_run.zip.")
    uploaded = colab_files.upload()
    archives = [(name, data) for name, data in uploaded.items()
                if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Please upload exactly one project ZIP file.")
    destination = Path.cwd().resolve()
    with ZipFile(io.BytesIO(archives[0][1])) as archive:
        expected = {
            "bioreactor_ready_to_run/run.py",
            "bioreactor_ready_to_run/requirements.txt",
            "bioreactor_ready_to_run/lab/common.py",
        }
        if not expected.issubset(set(archive.namelist())):
            raise ValueError("This ZIP does not contain the expected project folder.")
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination / "bioreactor_ready_to_run"):
                raise ValueError("The ZIP contains a path outside the project folder.")
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("The ZIP contains an unsupported symbolic link.")
        archive.extractall(destination)
    ROOT = find_project()

if ROOT is None:
    raise FileNotFoundError(
        "Extract bioreactor_ready_to_run.zip, then open this notebook inside "
        "the bioreactor_ready_to_run folder and run this cell again."
    )

print("Project folder:", ROOT)
print("Selected mode:", MODE)


## Step 2. Install the required Python packages

Run this cell once in your current environment. It installs the versions listed in `requirements.txt`. Installation needs an internet connection.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")],
    check=True,
    cwd=ROOT,
)
print("The required packages are installed.")


## Step 3. Generate the results

The quick mode reads saved predictions. The training mode fits the models in a fresh run folder. Keep this cell running until it finishes.

Results will be available in the project's `outputs` folder. The main submission file is `submission_best.csv`.


In [ ]:
command = [sys.executable, str(ROOT / "run.py")]
if MODE == "retrain":
    command.append("--retrain")

print("Running mode:", MODE, flush=True)
subprocess.run(command, check=True, cwd=ROOT)
OUTPUTS = ROOT / "outputs"
print("Results folder:", OUTPUTS)


## Step 4. Check and download the results

This cell shows the output checks and the first five rows of the main submission. Each row contains a batch, a sampling time, and four predicted measurements.

In Google Colab, it also downloads the three submission CSV files. In local Jupyter, it shows links to those files. The run log and experiment report are available in `outputs`.


In [ ]:
import pandas as pd
from IPython.display import FileLink, display

with (OUTPUTS / "validation.json").open(encoding="utf-8") as handle:
    validation = json.load(handle)
print("Output validation:")
print(json.dumps(validation, indent=2))

submission = pd.read_csv(OUTPUTS / "submission_best.csv", dtype={"batch_id": str})
print("Submission rows:", len(submission))
display(submission.head())

filenames = [
    "submission_best.csv",
    "submission_nosearch.csv",
    "submission_mechanistic_blend.csv",
]
for filename in filenames:
    output_file = OUTPUTS / filename
    if not output_file.is_file():
        raise FileNotFoundError(f"Expected output is missing: {output_file}")
    if colab_files is not None:
        colab_files.download(str(output_file))
    else:
        display(FileLink(str(output_file.relative_to(Path.cwd()))))
